# Evaluating a function

This section describes how to evaluate a `RealFunction` or `ComplexFunction` at an array of points.

## Theory

Evaluating a Chebyshev series of the form
```{math}
:label: chebyshev_series
f(x) = \sum_{n=0}^{N} c_n T_n[\xi(x)] \;,
```
at a set of $M$ points $x_m$ can be done naively by evaluating the Chebyshev polynomials $T_n$ at each point $x_m$ and summing the results.
Using the recurrence relation {eq}`rec_cheby`, the complexity of evaluating $T_n[\xi(x_m)]$ for all $n$ and $m$ is $(2N-3)M$.
Then the cost of evaluating the sum for all $M$ points is $(2N+1)M$.

Instead, `cheby` uses a more efficient method (both in terms of operations and memory), which is the Clenshaw recurrence {cite:p}`boyd01`.
Starting from $b_{N+1} = b_{N} = 0$, the recurrence reads
```{math}
:label: clenshaw
b_k = 2\xi\,b_{k+1} - b_{k+2} + c_k \;, \quad k = N, N-1, \dots, 1
\;,
```
with $\xi$ the mapped coordinate given by {eq}`xi_map`.
The result is then recovered from
```{math}
f(x) = \xi\,b_1 - b_2 + c_0 \;,
```
This is essentially a Horner-like scheme adapted to the three-term recurrence of the Chebyshev polynomials.

## Python API

Let us first create a `RealFunction` object representing the function $f(x) = \sin(3 x)\ee^{-0.3 x}$ on the interval $[0, 5]$.

In [ ]:
import numpy as np
from cheby import RealFunction

def f_exact(x):
    return np.sin(3 * x) * np.exp(-0.3 * x)

f = RealFunction(f_exact, 0.0, 5.0)

We can then evaluate the function at an array of points by calling the object `f` directly, i.e. `f(x)` where `x` is a NumPy array of points.
The evaluation uses the Clenshaw recurrence described above.

In [ ]:
import matplotlib.pyplot as plt
%config InlineBackend.figure_formats = ["svg", "pdf"]

x = np.linspace(0.0, 5.0, 300)
y = f(x)

plt.figure()
plt.plot(x, y, label='Chebyshev series')
plt.plot(x, f_exact(x), '--', label='Original function')
plt.legend()
plt.xlabel(r'$x$')
plt.show()

We can calculate the error between the original function and the Chebyshev series approximation:

In [ ]:
print('Largest difference:', np.max(np.abs(y - f_exact(x))))

We can see that the Chebyshev series approximation is very accurate.

As a consistency check, we can evaluate the Chebyshev polynomials at the Chebyshev nodes and then compute the sum {eq}`chebyshev_series`.
This should give the same result as the Clenshaw recurrence.

In [ ]:
Tn = f.basis().eval(x)
y_manual = Tn @ f.coef
print('Largest difference:', np.max(np.abs(y - y_manual)))